# Retail Sales Intelligence & Forecasting Platform
### End-to-End Analytics, Star Schema SQL Data Warehouse & Short-Term Sales Forecasting

**Author:** Data Science & Analytics Team  
**Target Architecture:** Python (ETL & ML) &bull; MySQL / Relational Data Warehouse &bull; SAP Analytics Cloud (SAC)  
**Dataset:** 24-Month Validated Retail Transactions (2024-01-01 to 2025-12-31)

---

## 1. Setup & Environment Initialization

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root in python path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import CLEAN_DATA_FILE
from src.analytics.queries import get_executive_kpis, get_monthly_trends, get_category_performance
from src.analytics.reconciliation import reconcile_kpis

print("Environment initialized successfully.")

## 2. Load Validated Dataset & Executive Performance Metrics

In [ ]:
df = pd.read_csv(CLEAN_DATA_FILE)
df["order_date"] = pd.to_datetime(df["order_date"])

print(f"Total Transactions: {len(df):,}")
print(f"Distinct Orders:    {df['order_id'].nunique():,}")
print(f"Distinct Customers: {df['customer_id'].nunique():,}")
print(f"Total Revenue:     ${df['revenue'].sum():,.2f}")
print(f"Total Net Profit:  ${df['profit'].sum():,.2f}")
print(f"Profit Margin:     {(df['profit'].sum() / df['revenue'].sum()) * 100:.2f}%")
print(f"Average Order Value: ${df['revenue'].sum() / df['order_id'].nunique():,.2f}")

df.head()

## 3. SQL vs Python KPI Reconciliation
Validating that Star Schema aggregations reconcile with 100% precision.

In [ ]:
rec_result = reconcile_kpis()
assert rec_result["reconciled"] is True

## 4. Monthly Trajectory & Seasonality Analysis

In [ ]:
monthly = df.groupby(pd.Grouper(key="order_date", freq="ME")).agg(
    revenue=("revenue", "sum"),
    profit=("profit", "sum")
).reset_index()
monthly["month_str"] = monthly["order_date"].dt.strftime("%b %y")

plt.figure(figsize=(12, 5))
plt.plot(monthly["month_str"], monthly["revenue"] / 1000, marker="o", label="Revenue ($K)", color="#0070f3", lw=2)
plt.plot(monthly["month_str"], monthly["profit"] / 1000, marker="s", label="Profit ($K)", color="#059669", lw=2)
plt.fill_between(monthly["month_str"], monthly["profit"] / 1000, color="#059669", alpha=0.15)
plt.title("Historical Monthly Sales & Profitability Trajectory (2024 - 2025)", fontsize=13, fontweight="bold")
plt.xlabel("Month")
plt.ylabel("Amount ($K)")
plt.xticks(rotation=45)
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## 5. Discount Impact & Margin Erosion
Testing the business question: *How do discounts relate to profitability?*

In [ ]:
plt.figure(figsize=(9, 5))
tier_order = ["No Discount (0%)", "Low (1-10%)", "Medium (11-20%)", "High (>20%)"]
sns.boxplot(data=df, x="discount_tier", y="profit_margin_pct", order=tier_order, palette=["#059669", "#0070f3", "#d97706", "#e11d48"])
plt.axhline(0, color="#e11d48", linestyle="--", label="Breakeven (0% Margin)")
plt.title("Profit Margin Distribution by Discount Tier", fontsize=13, fontweight="bold")
plt.xlabel("Discount Tier")
plt.ylabel("Profit Margin (%)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

## 6. Short-Term 3-Month Sales Forecasting
Evaluating chronological machine learning model vs naive baselines.

In [ ]:
from src.forecasting.model import evaluate_and_forecast
results = evaluate_and_forecast()
results["forecast"]